# Testing Knowledge Graph and LLM model

## Knowledge Graph

Rancho has ingested OpenTargets into a neo4j knowledge graph using biocypher package.

To access the graph, you'll need credentials. These are supplied via `.env` file that should have **NEO4J_URI**, **NEO4J_USERNAME** and **NEO4J_PASSWORD** in it. Example URI is "bolt+s://neo4j.myhost.com:7687"


In [10]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the variables from the environment
neo4j_uri = os.getenv("NEO4J_URI")
neo4j_username = os.getenv("NEO4J_USERNAME")
neo4j_password = os.getenv("NEO4J_PASSWORD")

# Check if any variable is missing
if not all([neo4j_uri, neo4j_username, neo4j_password]):
    raise EnvironmentError("One or more environment variables are missing: NEO4J_URI, NEO4J_USERNAME, NEO4J_PASSWORD")

print(f"Accessing OpenTargets at {neo4j_uri} as user {neo4j_username}")

Accessing OpenTargets at bolt+s://pistoia.neo4j.rbsapp.net:7687 as user neo4j


Here are a couple of examples on how to run arbitrary queries on Neo4j. 

First, using **neo4j** library (pip install neo4j)


In [11]:
from neo4j import GraphDatabase

# Create a driver instance
driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_username, neo4j_password))

# Define a function to run your queries
def run_query():
    with driver.session() as session:
        # Execute a Cypher query
        result = session.run("MATCH (n) RETURN n LIMIT 10")
        out = []
        # Iterate over the result and print records
        for record in result:
            out.append(record)
            print(record)
    return out

# Call the function
results = run_query()

# Close the driver when done
driver.close()

print(results)

<Record n=<Node element_id='4:2714e891-0c75-41c7-9118-5df655f3fe75:0' labels=frozenset({'ThingWithTaxon', 'BiologicalEntity', 'Entity', 'NamedThing', 'Obi.Disease', 'DiseaseOrPhenotypicFeature', 'Disease'}) properties={'licence': 'https://platform-docs.opentargets.org/licence', 'code': 'http://purl.obolibrary.org/obo/OBI_1110122', 'name': 'pathological process', 'preferred_id': 'obi', 'description': 'Abnormal, harmful processes caused by or associated with a disease', 'source': 'Open Targets', 'id': 'obi:1110122', 'version': '22.11'}>>
<Record n=<Node element_id='4:2714e891-0c75-41c7-9118-5df655f3fe75:1' labels=frozenset({'ThingWithTaxon', 'BiologicalEntity', 'Entity', 'NamedThing', 'Obi.Disease', 'DiseaseOrPhenotypicFeature', 'Disease'}) properties={'licence': 'https://platform-docs.opentargets.org/licence', 'code': 'http://purl.obolibrary.org/obo/OBI_0001621', 'name': 'longitude', 'preferred_id': 'obi', 'description': 'A measurement that is the measure of the longitude coordinate of 

Another way - py2neo library - provides a higher level interface (pip install py2neo)

In [12]:
from py2neo import Graph

# Create a Graph instance
graph = Graph(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password)
)

# Run a query
results = graph.run("MATCH (n) RETURN n LIMIT 10")

# Print the results
for record in results:
    print(record)

Node('BiologicalEntity', 'Disease', 'DiseaseOrPhenotypicFeature', 'Entity', 'NamedThing', 'Obi.Disease', 'ThingWithTaxon', code='http://purl.obolibrary.org/obo/OBI_1110122', description='Abnormal, harmful processes caused by or associated with a disease', id='obi:1110122', licence='https://platform-docs.opentargets.org/licence', name='pathological process', preferred_id='obi', source='Open Targets', version='22.11')
Node('BiologicalEntity', 'Disease', 'DiseaseOrPhenotypicFeature', 'Entity', 'NamedThing', 'Obi.Disease', 'ThingWithTaxon', code='http://purl.obolibrary.org/obo/OBI_0001621', description='A measurement that is the measure of the longitude coordinate of a site.', id='obi:0001621', licence='https://platform-docs.opentargets.org/licence', name='longitude', preferred_id='obi', source='Open Targets', version='22.11')
Node('BiologicalEntity', 'Disease', 'DiseaseOrPhenotypicFeature', 'Entity', 'NamedThing', 'Obi.Disease', 'ThingWithTaxon', code='http://purl.obolibrary.org/obo/OBI_0

## LLM models

We'll use Langchain framework to invoke LLM models from OpenAI, Anthropic, Mistral and other vendors.

To work with LLMs, you need to provide API keys in .env file - **OPENAI_API_KEY**, **ANTHROPIC_API_KEY**, **MISTRAL_API_KEY**.

A few examples to highlight the capabilities of LLMs in generating Cypher queries:

In [13]:

prompt = "I have a Neo4j graph with biological data that was extracted from OpenTargets using biocypher. Generate a cypher query that would help me understand what's in the graph"


In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = "gpt-4o")
result = llm.invoke(prompt)
print(result.content)

To understand the contents of your Neo4j graph, particularly when dealing with biological data extracted from OpenTargets using Biocypher, you can execute several Cypher queries that give you insights into the schema, node types, relationships, and properties of your graph. Here are some example queries you can run:

1. **List All Node Labels**: This will give you an overview of the different types of nodes present in your graph.

    ```cypher
    CALL db.labels()
    ```

2. **List All Relationship Types**: Understanding the types of relationships is crucial for understanding how nodes are interconnected.

    ```cypher
    CALL db.relationshipTypes()
    ```

3. **Count Nodes by Label**: This query helps you see how many nodes exist for each label, which can give insights into the distribution of data.

    ```cypher
    CALL db.labels() YIELD label
    RETURN label, count(*) AS count
    ORDER BY count DESC
    ```

4. **Count Relationships by Type**: Similar to nodes, it’s useful 

In [15]:
# 01-preview is quite expensive. 

# temperature=1 is apparently required for this model to work

llm = ChatOpenAI(model = "o1-preview-2024-09-12", temperature = 1)
result = llm.invoke(prompt)
print(result.content)

To help you understand the contents of your Neo4j graph extracted from OpenTargets using Biocypher, you can run a series of Cypher queries that explore the structure and data within the graph. These queries will provide insights into the types of nodes and relationships present, their properties, and how they're connected.

---

### **1. Get a Summary of Node Labels and Counts**

This query retrieves all node labels in the graph along with the count of nodes for each label.

```cypher
MATCH (n)
RETURN labels(n) AS NodeLabels, COUNT(*) AS Count
ORDER BY Count DESC
```

**What it does:**
- `MATCH (n)` finds all nodes.
- `labels(n)` retrieves the labels of each node.
- Groups nodes by their labels and counts them.
- Orders the result by the count in descending order.

---

### **2. Get a Summary of Relationship Types and Counts**

This query lists all relationship types and the number of times each occurs in the graph.

```cypher
MATCH ()-[r]->()
RETURN TYPE(r) AS RelationshipType, COUNT(

In [ ]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model = "claude-3-5-sonnet-20240620")
result = llm.invoke(prompt)
print(result.content)

In [ ]:
from langchain_mistralai import ChatMistralAI

# open-mistral-7b is their "legacy" model. It could be deployed locally, but we'll use API to query

llm = ChatMistralAI(model = "open-mistral-7b")
result = llm.invoke(prompt)
print(result.content)